## Merging data from gamelist.xml files

in hopes that there is some missing stuff

- gamelist.xml files are in lists/<platform_name>/gamelist.xml
- platform name mappings in lists/systems.
- determine gamelist.xml tags to determine mapping
- prepare data from gamelist.xml files (to csv?)
- merge data to main

In [1]:
from utils.gamelist_parser import load_all_gamelists
from utils.merge_pipeline import run_merge_pipeline, SourceConfig

### Step 1: Parse all gamelist.xml files

In [2]:
# Load and parse all gamelist.xml files
gamelist_df = load_all_gamelists(lists_dir="lists", systems_file="utils/platform_mappings.json")
gamelist_df.head()

217 games from n64 (Nintendo 64)
7 games from zxspectrum (Sinclair ZX Spectrum)
10 games from dreamcast (Sega Dreamcast)
265 games from psx (Sony Playstation)
564 games from msx (Microsoft MSX)
0 games from xbox (Microsoft Xbox)
178 games from pc (MS-DOS)
29 games from c64 (Commodore 64)
67 games from ps2 (Sony Playstation 2)
548 games from gbc (Nintendo Game Boy Color)
1204 games from gba (Nintendo Game Boy Advance)
574 games from tg16 (NEC TurboGrafx-16)
6 games from doom (MS-DOS)
189 games from neogeo (SNK Neo Geo)
263 games from gamegear (Sega Game Gear)
75 games from nds (Nintendo DS)
976 games from snes (Super Nintendo Entertainment System)
842 games from fba (Arcade)
290 games from pcengine (NEC PC Engine)
20 games from scummvm (ScummVM Game Engine)
1928 games from mame (Multiple Arcade Machine Emulator)
1024 games from genesis (Sega Genesis)
81 games from ngpc (SNK Neo Geo Pocket Color)
1948 games from famicom (Nintendo Entertainment System)
493 games from gb (Nintendo Game Boy

,platform,name,filename,summary,release_date,developer,publisher,genres,players,user_rating
0,Nintendo 64,007 : The World is Not Enough,007 - The World Is Not Enough,The World Is Not Enough is a first-person shoo...,20001101T000000,Eurocom Developments,Electronic Arts,Shooter / 1St Person,1-4,6.5
1,Nintendo 64,1080 TenEighty Snowboarding,1080 Snowboarding (JU) [!],1080Â° Snowboarding is a snowboard racing game...,19980401T000000,Nintendo,Nintendo,Sports / Skiing,1-2,8.5
2,Nintendo 64,40 Winks,40 Winks (Proto),There's only 40 Winks left in the world of dre...,19991114T000000,Eurocom,Piko Interactive,Action,1-2,8.0
3,Nintendo 64,A Bug's Life,"A, Bug's Life","Based on the animated film, A Bug's Life is an...",19990430T000000,Travellersâ Tales,Activision,Platform,None,6.0
4,Nintendo 64,Aerofighter Assault,AeroFighters Assault,The AeroFighters Assault team needs your help ...,19971030T000000,Paradigm Entertainment,Video System,Shooter / Plane,1-2,7.0


In [3]:
# Check data quality
print(gamelist_df['platform'].unique())
print(gamelist_df.columns)

['Nintendo 64' 'Sinclair ZX Spectrum' 'Sega Dreamcast' 'Sony Playstation'
 'Microsoft MSX' 'MS-DOS' 'Commodore 64' 'Sony Playstation 2'
 'Nintendo Game Boy Color' 'Nintendo Game Boy Advance' 'NEC TurboGrafx-16'
 'SNK Neo Geo' 'Sega Game Gear' 'Nintendo DS'
 'Super Nintendo Entertainment System' 'Arcade' 'NEC PC Engine'
 'ScummVM Game Engine' 'Multiple Arcade Machine Emulator' 'Sega Genesis'
 'SNK Neo Geo Pocket Color' 'Nintendo Entertainment System'
 'Nintendo Game Boy' 'Sega Master System' 'Sega CD' 'Commodore Amiga'
 'Sega Saturn' 'Sony Playstation Portable' 'Super Famicom'
 'NEC PC Engine CD' 'Atari 2600' 'Nintendo Wii' 'Sega Mega Drive'
 'Nintendo Famicom Disk System' 'Nintendo GameCube']
Index(['platform', 'name', 'filename', 'summary', 'release_date', 'developer',
       'publisher', 'genres', 'players', 'user_rating'],
      dtype='object')


In [4]:
gamelist_df.isnull().sum()

platform           0
name               0
filename           0
summary          528
release_date    1389
developer       1243
publisher        615
genres          1059
players         2492
user_rating     1475
dtype: int64

### Step 2 gamelist source config

In [5]:
gamelist_config = SourceConfig(
    name="gamelist",
    loader=lambda: gamelist_df
)

### Step 3: Integrate with main

In [6]:
# Option B: Load existing merged data and add gamelist source
import pickle

with open('merged_df.pkl', 'rb') as f:
    merged_df = pickle.load(f)

print(f"Existing merged data: {len(merged_df)} games")
print(f"Gamelist data: {len(gamelist_df)} games")

Existing merged data: 206749 games
Gamelist data: 16258 games


In [7]:
# Merge into existing data
final_merged = run_merge_pipeline(
    main_config=SourceConfig(
        name="merged_base",
        loader=lambda: merged_df
    ),
    source_configs=[gamelist_config]
)

print(f"\nFinal merged data: {len(final_merged)} games")
print(f"Added {len(final_merged) - len(merged_df)} new games")

gamelist length: 16258, main length: 214802

Final merged data: 214802 games
Added 8053 new games


In [8]:
# show the new games that were added
new_games = final_merged[~final_merged.set_index(['name', 'platform']).index.isin(
    merged_df.set_index(['name', 'platform']).index
)]
new_games.head()

,name,platform,filename,summary,release_date,release_year,genres,developer,publisher,players,cooperative,rating,user_rating
53,'88 Games,Multiple Arcade Machine Emulator,88games,Konami '88 (also known as '88 Games or Hyper S...,1988-01-01,<NA>,Esporte-Esporte / Pista de Corrida,Konami,Konami,2,NaN,<NA>,4.0
154,005,Multiple Arcade Machine Emulator,005,"Neste jogo de furtividade, o objetivo é atrave...",1981-01-01,<NA>,Shooter Small-Ação / Labirinto-Ação,Sega,Sega,1-2,NaN,<NA>,10.0
185,007: Everything Or Nothing,Nintendo Game Boy Advance,"007 - Everything or Nothing (USA, Europe) (En,...","Think like Bond, act like Bond, and experience...",2003-11-17,<NA>,Tiro / Tiro em 3ª pessoa-Tiro,Griptonite Games,Electronic Arts,2,NaN,<NA>,8.0
207,007: Nightfire,Nintendo Game Boy Advance,"007 - NightFire (USA, Europe) (En,Fr,De)",The ultimate secret agent is back in his most ...,2003-03-17,<NA>,Esporte-Tiro / Tiro em 1ª pessoa-Tiro,JV Games,Electronic Arts,2,NaN,<NA>,6.0
218,007: The World Is Not Enough,Nintendo Game Boy Color,"007 - The World Is Not Enough (USA, Europe)",It seems that an MI-6 agent has been killed ju...,2001-09-11,<NA>,Ação,2n Productions,Electronic Arts,2,NaN,<NA>,4.0


In [9]:
final_merged.head()

,name,platform,filename,summary,release_date,release_year,genres,developer,publisher,players,cooperative,rating,user_rating
0,! That Bastard Is Trying To Steal Our Gold !,Windows,NaN,Steal gold from the Lerpikon's dungeons! Get r...,NaT,<NA>,"Platform, Puzzle",WTFOMGames,WTFOMGames,NaN,False,<NA>,8.000000
1,!!!Ants!!!,Tandy TRS-80,NaN,In Ants you can become an ant and join the ran...,NaT,1979,NaN,Brian Rotolante,Synergistic Solar Inc.,1.0,False,<NA>,5.000000
2,"""300""",Pinball,NaN,"""300"" (the exact machine name includes the quo...",NaT,1975,Pinball,Gottlieb,Gottlieb,4.0,False,<NA>,6.944444
3,"""8 Ball""",Pinball,NaN,A pool table themed pinball game from Williams...,NaT,1952,Pinball,Williams Electronic Games,Williams Electronic Games,2.0,False,<NA>,7.250000
4,"""History in the Making"": The First Three Years",Amstrad CPC,NaN,A celebration of U.S. Gold's first three years...,NaT,1988,Compilation,NaN,U.S. Gold,4.0,False,<NA>,6.972222


## Step 5: Save updated merged data

In [10]:
# Save the updated merged data
with open('final_merged_df.pkl', 'wb') as f:
    pickle.dump(final_merged, f)

print("Saved merged data to merged_df.pkl")

Saved merged data to merged_df.pkl
